# La base de connaissances d'un débat — propositions, arguments, conflits

> **Série ArgumentAnalysis.** Ce carnet distille le module `knowledge_base` du moteur de débat : la structure de données qui enregistre *ce qui a été dit* dans un débat formel — les propositions posées, les arguments avancés, et le conflit ouvert entre eux. Il complète la série en descendant d'un cran : `Toulmin_Model` déplie la structure d'UN argument, `Schemes_Walton` identifie sa forme stéréotypée, `Dialogues_Protocolises` organise qui parle quand — la base de connaissances est la **mémoire commune** où tous ces actes laissent une trace.

**Provenance et contrat de fidélité.** Le module distillé vient du moteur étudiant `1_2_7_argumentation_dialogique/local_db_arg/src/core/knowledge_base.py` — chemin toujours présent dans le dépôt tronc. Il avait été restauré un temps dans le paquet `argumentation_analysis/agents/core/debate/` (G8 #1184), puis ce chemin a été retiré du tronc par le triage #2137 (PR #2273) : ce carnet en assume désormais la conservation pédagogique, en le disant. Le port C# préserve le comportement et documente quatre divergences mesurées au lieu de les corriger en silence : `rules` et `preferences` (état mort jamais lu, non portés), `Entails` (un test d'appartenance, pas un moteur d'inférence), la négation purement lexicale (`¬¬P` n'est pas `P`), et l'écrasement des propositions homonymes. La règle du dépôt s'applique : on restaure, on ne fabrique pas.

L'organe est **déterministe** : aucun LLM, aucune JVM, aucun aléatoire qui affecte un résultat. Tout ce que ce carnet affiche est rejouable à l'identique. Les exemples sont synthétiques et domaine-public — le banc de démonstration est celui du carnet source EPITA (deux camps opposés sur la concentration en télétravail).

Dans le moteur étudiant d'origine, cette base est le socle sur lequel tournent les stratégies de débat : chaque tour de dialogue interroge `Support` pour savoir ce qui peut être avancé, et `IsConsistent` pour savoir si le désaccord persiste. Le carnet n'emporte pas ces stratégies (elles appellent des LLM — c'est précisément le bruit que la distillation retire) : il isole la **mémoire**, pour qu'on puisse l'étudier sans dépendance. C'est la même discipline que pour les schémas de Walton : l'objet pédagogique est la structure déterministe, pas le service qui la consomme.

Ce que couvre ce carnet :

1. **§1** la population transitive — l'argument porte ses prémisses et sa conclusion ;
2. **§2** support et attaque — la convention lexicale de négation `¬` ;
3. **§2 bis** la mémoire vivante — le lien dirigé `argument_network` que le tronc a retenu pour la limite mesurée en §2 ;
4. **§3** la cohérence — P et `¬P` cohabitants signalent un débat ouvert, et le nommage honnête de `Entails` ;
5. **§4** les limites mesurées — non-inférence, double négation, écrasement ;
6. trois exercices.

**Objectifs :** construire une mémoire, distinguer soutien, attaque et cohérence. **Prérequis :** classes et dictionnaires C#. **Durée :** 45 minutes.

**Série Argumentation** | | Précédent : [04-Dialogues-Protocolises-CSharp](./Argumentation-04-Dialogues-Protocolises-CSharp.ipynb) | Suivant : [05-Formal-Verification-Python](./Argumentation-05-Formal-Verification-Python.ipynb) | [↑ Série Argumentation](README.md)

## §1 — Le modèle : population transitive

La base tient en deux dictionnaires : `propositions` (indexé par contenu) et `arguments` (indexé par identifiant). La méthode centrale est `AddArgument` : elle enregistre l'argument **et** enregistre chacune de ses prémisses et sa conclusion comme propositions. C'est ce que nous appelons la **population transitive** — personne n'a besoin d'ajouter « Le télétravail isole les collaborateurs » explicitement : l'argument qui la cite comme prémisse la porte en base.

In [1]:
// Identité lexicale : seules les chaînes de contenu définissent l'égalité.
using System;
using System.Collections.Generic;
using System.Linq;

class Proposition : IEquatable<Proposition>
{
    public string Content;
    public bool? TruthValue;
    public double Confidence;
    public string Source;
    public Proposition(string content, bool? truthValue = null, double confidence = 1.0, string source = null)
    {
        Content = content;
        TruthValue = truthValue;
        Confidence = confidence;
        Source = source;
    }

    public bool Equals(Proposition p) => p is not null && Content == p.Content;
    public override bool Equals(object p) => p is Proposition x && Equals(x);
    public override int GetHashCode() => Content.GetHashCode();
    public override string ToString() => Content;
}

class FormalArgument
{
    public List<Proposition> Premises;
    public Proposition Conclusion;
    public double Strength;
    public string Scheme, Id;
    public FormalArgument(IEnumerable<Proposition> premises, Proposition conclusion, double strength = 1, string scheme = null, string id = null)
    {
        Premises = premises.ToList();
        Conclusion = conclusion;
        Strength = strength;
        Scheme = scheme;
        Id = string.IsNullOrEmpty(id) ? Guid.NewGuid().ToString() : id;
    }

    public override string ToString() => $"[{string.Join(", ", Premises)}] -> {Conclusion}";
}

class KnowledgeBase
{
    public Dictionary<string, Proposition> Propositions = new();
    public Dictionary<string, FormalArgument> Arguments = new();
    public void AddProposition(Proposition p) => Propositions[p.Content] = p;
    public void AddArgument(FormalArgument a)
    {
        Arguments[a.Id] = a;
        foreach (var p in a.Premises)
            AddProposition(p);
        AddProposition(a.Conclusion);
    }

    public List<FormalArgument> Support(Proposition p) => Arguments.Values.Where(a => a.Conclusion.Content == p.Content).ToList();
    public List<FormalArgument> Attack(Proposition p) => Arguments.Values.Where(a => a.Conclusion.Content == "¬" + p.Content).ToList();
    public bool IsConsistent() => !Propositions.Keys.Any(k => Propositions.ContainsKey("¬" + k));
    public bool Entails(Proposition p) => Propositions.ContainsKey(p.Content);
}

Proposition Negation(Proposition p) => new("¬" + p.Content);
KnowledgeBase Construire(bool second)
{
    var k = new KnowledgeBase();
    k.AddArgument(new(new[] { new Proposition("Le télétravail isole les collaborateurs") }, new("Le télétravail réduit la concentration"), scheme: "cause_effect"));
    if (second)
        k.AddArgument(new(new[] { new Proposition("Les pauses régulières restaurent la concentration") }, new("¬Le télétravail réduit la concentration")));
    return k;
}

var kb = Construire(true);
var kbSaine = Construire(false);
Console.WriteLine($"Propositions enregistrées ({kb.Propositions.Count}) : {string.Join(" ; ", kb.Propositions.Keys)}");
Console.WriteLine($"Arguments enregistrés : {kb.Arguments.Count}");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Propositions enregistrées (4) : Le télétravail isole les collaborateurs ; Le télétravail réduit la concentration ; Les pauses régulières restaurent la concentration ; ¬Le télétravail réduit la concentration


Arguments enregistrés : 2


**Lecture du résultat.** Quatre propositions pour deux arguments : les deux conclusions (« Le télétravail réduit la concentration » et sa négation préfixée « ¬Le télétravail réduit la concentration ») et les deux prémisses (« ...isole les collaborateurs », « Les pauses régulières restaurent la concentration »). Aucune des deux prémisses n'a été ajoutée par un `AddProposition` explicite — c'est `AddArgument` qui les a portées. Un décompte qui mérite l'œil : deux arguments, chacun portant une prémisse et une conclusion, cela fait quatre ajouts — et exactement quatre propositions. Aucun doublon ne s'est déclaré parce qu'aucune prémisse n'est partagée entre les deux arguments ; le §4 montrera ce qui arrive quand deux propositions de même contenu se présentent. Notez la convention déjà visible dans la liste : la négation n'est pas un champ ou un booléen, c'est le **préfixe de chaîne `¬`** collé au contenu — nous y revenons au §2.

In [2]:
// Une prémisse entre en base avec son argument, contrairement à l'intrus.
var premisse = new Proposition("Le télétravail isole les collaborateurs");
var intrus = new Proposition("Le télétravail augmente la productivité");
Console.WriteLine($"entails(premisse) = {kb.Entails(premisse)} ; entails(intrus) = {kb.Entails(intrus)}");

entails(premisse) = True ; entails(intrus) = False


**Lecture du résultat.** `True` pour la prémisse portée, `False` pour l'intrus. Le test `Entails` ne fait qu'une chose — vérifier l'appartenance au dictionnaire des propositions — mais la population transitive lui donne déjà un intérêt : il distingue ce qui **a été dit** (posé ou porté par un argument) de ce qui ne l'a pas été. La distinction est opérationnelle, pas philosophique : un agent de dialogue qui prépare son prochain coup demande à la mémoire « puis-je m'appuyer sur cette prémisse ? » — la réponse dépend de si elle a été posée, pas de si elle est vraie. C'est une mémoire de débat, pas une théorie logique : le §4 mesurera précisément ce que cette mémoire ne sait pas faire, notamment quand une conclusion « devrait » suivre sans qu'aucun argument ne l'ait jamais conclue.

## §2 — Support et attaque : qui conclut quoi

Deux requêtes parcourent les arguments : `find_supporting_arguments(P)` renvoie ceux dont la **conclusion est exactement P**, `find_attacking_arguments(P)` ceux dont la conclusion est **`"¬" + contenu de P`**. La convention est lexicale — la négation est une construction de chaîne, pas un opérateur — et elle a une conséquence directe sur ce qu'une « attaque » est ici : **un fait de conclusion, pas un acte dirigé**. La base ne sait pas QUEL argument répond à quel autre, ni dans quel ordre ; elle sait seulement que la négation de P a été conclue quelque part. Le port expose le helper `Negation()` pour rendre cette convention explicite et testable (le tronc construit la chaîne inline à chaque appel).

In [3]:
// Les conclusions égales à P soutiennent ; celles égales à ¬P attaquent.
var prop = new Proposition("Le télétravail réduit la concentration");
var soutiens = kb.Support(prop);
var attaques = kb.Attack(prop);
Console.WriteLine($"Soutiens : {soutiens.Count} ; attaques : {attaques.Count}");
foreach (var a in soutiens.Concat(attaques))
    Console.WriteLine($"{a} (schéma : {a.Scheme ?? "null"})");

Soutiens : 1 ; attaques : 1


[Le télétravail isole les collaborateurs] -> Le télétravail réduit la concentration (schéma : cause_effect)


[Les pauses régulières restaurent la concentration] -> ¬Le télétravail réduit la concentration (schéma : null)


**Lecture du résultat.** Un soutien, une attaque — le débat est équilibré. Le soutien conclut exactement P ; l'attaque conclut exactement `¬P` (préfixe compris). Deux détails mesurables : le schéma est `cause_effect` sur le soutien et `null` sur l'attaque (le moteur étudiant ne labelait pas tous ses arguments), et le rendu `str` de l'argument affiche `[prémisses] -> conclusion` — c'est la convention d'affichage du port, pratique pour lire une base d'un coup d'œil. L'attaque ne **réfute** rien : elle coexiste — c'est le §3 qui dira si cette coexistence est un conflit.

In [4]:
// Même requête sur une base à un seul camp.
Console.WriteLine($"Base saine — soutiens : {kbSaine.Support(prop).Count}, attaques : {kbSaine.Attack(prop).Count}");

Base saine — soutiens : 1, attaques : 0


**Lecture du résultat.** Même proposition, même requête, autre base : cette fois aucune attaque — personne n'a conclu `¬P`. La structure de la question ne change pas, c'est l'état de la mémoire qui diffère — et c'est la leçon de méthode : interroger une base de connaissances ne révèle jamais que son état courant, jamais l'historique qui l'a produit. C'est tout ce que « attaquer » veut dire ici : avoir déjà conclu la négation de la cible. Une attaque n'est pas un acte daté ni un lien dirigé entre arguments (pour cela, voir les cadres d'argumentation abstraits de `Dung_AF_Semantics`, où les attaques sont des arêtes nommées d'un nœud vers un autre) — c'est un fait statistique sur les conclusions enregistrées.

## Exercices

> **Convention C.1** : les stubs s'exécutent sans erreur (jamais `raise`). Remplir le corps, re-exécuter, vérifier. La méthode de chaque exercice est déjà démontrée dans le carnet :

- **Exercice 1** (compter les soutiens) → la section 2 l'a fait sur le télétravail : la même requête rend `Soutiens : 1` et `Attaques : 1` ; construire trois arguments dont deux concluent la même proposition, c'est répéter ce geste d'énumération.
- **Exercice 2** (ouvrir un débat en un seul geste) → la section 3 l'a mesuré : `deux_camps is_consistent -> False` quand `un_seul_camp -> True` ; un seul `AddArgument` dont la conclusion est la négation `¬` d'une proposition déjà présente fait basculer le verdict.
- **Exercice 3** (prédire avant de tester) → la section 4 a montré les pièges à prédire : aucun entaillement depuis la contradiction (`entails(arbitraire) = False`), `¬¬P` distinct de `P` — la prédiction se fait sur les mécanismes, pas sur l'intuition.

### Exercice 1 — Compter les soutiens d'une proposition

Construire une base de **trois** arguments dont **deux** concluent `« Le tramway réduit les embouteillages »` (prémisses au choix) et un conclut autre chose, puis afficher le nombre de soutiens de cette proposition.

In [5]:
// Exercice 1 : trois arguments, deux concluent le tramway réduit les embouteillages.
var maBase = new KnowledgeBase();
var cible = new Proposition("Le tramway réduit les embouteillages");
// TODO étudiant : ajouter les trois arguments puis compter Support(cible).
Console.WriteLine("Exercice à compléter");

Exercice à compléter


## §2 bis — La mémoire vivante : le lien dirigé que la base n'a pas

Le §2 a mesuré la limite : dans la base, une attaque est un **fait de conclusion** (`¬P` cohabite avec `P`), pas un acte dirigé —
rien ne dit quel argument répond à quel argument, ni dans quel ordre. Le tronc EPITA a retenu exactement cette structure ailleurs :
l'état vivant du débat, [`DebateState`](https://github.com/jsboigeEpita/2025-Epita-Intelligence-Symbolique/blob/ecfd9b9c31d299ad9e08ee0e7a726d436f4ebd8e/argumentation_analysis/agents/core/debate/debate_definitions.py#L77-L91)
porte le champ `argument_network: Dict[str, List[str]]` — le **graphe dirigé** des réponses. La cellule suivante le distille
(trois arguments nommés, deux arêtes) : c'est l'exact contraire du fait statistique du §2 — ici, l'arête **est** l'acte.
Contrairement aux cadres abstraits de `Dung_AF_Semantics` (des arêtes entre arguments abstraits), ce réseau vit dans la mémoire
du débat en cours, à côté des votes d'audience et de la phase — c'est ce qui permet aux stratégies du moteur de savoir à qui répondre.

In [6]:
// Graphe de réponses dirigées, distinct des conclusions lexicales de la base.
var network = new Dictionary<string, List<string>>
{
    {
        "B_rythme",
        new()
        {
            "A_teletravail"
        }
    },
    {
        "C_correctif",
        new()
        {
            "A_teletravail"
        }
    },
    {
        "A_teletravail",
        new()
    }
};
var cibleReseau = "A_teletravail";
Console.WriteLine($"Qui répond à {cibleReseau} ? {string.Join(", ", network.Where(e => e.Value.Contains(cibleReseau)).Select(e => e.Key))}");
foreach (var e in network)
    Console.WriteLine($"{e.Key} -> {e.Value.Count}");

Qui répond à A_teletravail ? B_rythme, C_correctif


B_rythme -> 1


C_correctif -> 1


A_teletravail -> 0


**Lecture du résultat.** L'arête dit ce que la base ne dira jamais : `B_rythme` **répond à** `A_teletravail` — un lien daté,
dirigé, interrogeable dans les deux sens (`qui répond à A ?`, `vers qui parle B ?`). La même information dans la base du carnet
serait deux conclusions opposées (`P` et `¬P`) sans aucun moyen de relier les arguments qui les portent. Le tronc a gardé les deux
organes séparés — la base pour la **cohérence** (§3), le réseau pour le **tour de parole** — et `DebateState` les réunit au moment
du débat : la limite mesurée en §2 n'est donc pas un défaut de conception, c'est la frontière exacte entre mémoire et orchestration.

## §3 — Cohérence, et un nommage honnête

`IsConsistent` répond `False` dès qu'une proposition **et** sa négation cohabitent dans la base — sans notion de degré : un conflit ouvert sur UNE proposition suffit à déclarer toute la base incohérente, même si cent autres propositions sont d'accord entre elles. La cohérence d'un débat est ici **binaire et globale**, exactement comme celle d'une théorie logique. Et `Entails` mérite qu'on le lise pour ce qu'il est : un test d'**appartenance** (`content in propositions`), pas un moteur d'inférence. La docstring du tronc le dit honnêtement (« contains ») ; la docstring du moteur étudiant promettait « implique » — le nom est plus ambitieux que la sémantique, et le savoir évite d'y brancher une attente déductive. Cette honnêteté de nommage est une leçon en soi : quand un nom promet plus que sa implémentation, le premier réflexe est de lire l'implémentation.

In [7]:
// Cohérence globale et trois requêtes littérales.
Console.WriteLine($"deux_camps is_consistent -> {kb.IsConsistent()} ; un_seul_camp -> {kbSaine.IsConsistent()}");
foreach (var s in new[]
{
    "Le télétravail isole les collaborateurs",
    "Le télétravail augmente la productivité",
    "¬Le télétravail réduit la concentration"
}

)
    Console.WriteLine($"entails({s}) = {kb.Entails(new(s))}");

deux_camps is_consistent -> False ; un_seul_camp -> True


entails(Le télétravail isole les collaborateurs) = True


entails(Le télétravail augmente la productivité) = False


entails(¬Le télétravail réduit la concentration) = True


**Lecture du résultat.** La base à deux camps est incohérente — P et `¬P` y cohabitent, le débat sur la concentration est **ouvert** ; la base à un camp est cohérente. C'est `IsConsistent` qui joue le rôle de détecteur de désaccord : une base de débat cohérente est une base où il ne reste rien à débattre. Le banc de requêtes confirme la lecture d'appartenance dans les deux sens : la négation `¬P` est « entraînée » au même titre que P — pour `Entails`, c'est une proposition comme une autre, aucune sémantique de conflit dans le test lui-même.

### Exercice 2 — Ouvrir un débat en un seul geste

Partir d'une base vide et la rendre **incohérente** avec un **seul** appel à `AddArgument`. Indice : la conclusion de l'argument porte sa propre négation... si la conclusion est `¬Q`, que faut-il avoir posé pour que `Q` et `¬Q` cohabitent ?

In [8]:
// Exercice 2 : ajouter un seul argument concluant la négation de Q.
var defi = new KnowledgeBase();
defi.AddProposition(new("La formation est utile"));
// TODO étudiant : ajouter l'argument, puis observer IsConsistent().
Console.WriteLine($"Exercice à compléter — base cohérente : {defi.IsConsistent()}");

Exercice à compléter — base cohérente : True


## §4 — Les limites mesurées : ce que la base ne sait PAS faire

Trois limites du modèle, **mesurées plutôt que supposées** : chacune est démontrée par une cellule exécutable ci-dessous, sur un cas où le comportement diverge de ce qu'une intuition logique classique prédit. Elles ne sont pas des défauts du port — chacune est un choix du moteur d'origine, documenté ici pour que l'utilisateur sache exactement ce qu'il tient (et ce qu'il ne tient pas). C'est la même discipline que pour les divergences des schémas de Walton : le port ne « répare » pas la sémantique mesurée, il la rend visible.

1. **`Entails` ne combine rien** — même depuis une contradiction, il n'entraîne pas une proposition arbitraire (pas de principe d'explosion).
2. **La négation est lexicale** — `¬¬P` n'est pas reconnu comme `P`.
3. **L'écrasement par contenu** — deux propositions homonymes ne laissent qu'un exemplaire.

In [9]:
// Absence d'explosion et absence de déduction arbitraire.
Console.WriteLine($"Incohérente : {!kb.IsConsistent()}, arbitraire : {kb.Entails(new("Une proposition quelconque jamais posée"))}");
var min = new KnowledgeBase();
min.AddProposition(new("A"));
Console.WriteLine($"Base {{A}} : entails(B) = {min.Entails(new("B"))}");

Incohérente : True, arbitraire : False


Base {A} : entails(B) = False


**Lecture du résultat.** La mémoire ne raisonne pas : elle regarde. Depuis `P` et `¬P` cohabitants — l'état le plus chargé d'une logique classique, d'où TOUT s'ensuit par explosion — le test d'appartenance répond toujours `False` pour une proposition non posée. La seconde moitié de la cellule montre l'autre face : une base réduite à `{A}` n'entraîne pas `B`, même si l'utilisateur « voit » un lien. C'est la frontière exacte entre **mémoire** et **moteur** : pour enchaîner prémisses et conclusions, il faut un vrai moteur — le solveur de la série `SMT/Z3-API` pour la déduction, ou les sémantiques de `Dung_AF_Semantics` pour calculer ce qui survit aux attaques. La base, elle, se contente de répondre « est-ce que ça a été dit ? ». Pour un débat d'opinion, cette modestie est une vertu : personne ne veut qu'une mémoire d'échanges fabrique des conclusions que personne n'a tirées.

In [10]:
// Double négation lexicale : ¬¬P reste une chaîne distincte de P.
var p0 = new Proposition("Le débat est ouvert");
var doubleNeg = Negation(Negation(p0));
var dn = new KnowledgeBase();
dn.AddProposition(p0);
dn.AddProposition(doubleNeg);
Console.WriteLine($"P={p0}, double={doubleNeg}, cohérente={dn.IsConsistent()}, présents={dn.Entails(p0) && dn.Entails(doubleNeg)}");

P=Le débat est ouvert, double=¬¬Le débat est ouvert, cohérente=True, présents=True


**Lecture du résultat.** P et `¬¬P` cohabitent sans conflit : `IsConsistent` cherche la paire **exacte** (contenu, `¬` + contenu), et `¬¬P` n'est le négation d'aucun des deux — la cellule affiche bien les DEUX propositions comme présentes (`Entails` répond deux fois vrai), et pourtant aucune incohérence. Classiquement, `¬¬P` est équivalent à `P` ; lexicalement, ce sont deux chaînes distinctes sans aucune relation. La convention est simple et vérifiable — mais elle impose une discipline à l'utilisateur : la forme exacte du contenu compte, caractère par caractère, y compris les espaces. Une base alimentée par des sources qui négativent différemment (« non-P », « il est faux que P ») ne détectera **aucun** de ces conflits — normaliser les énoncés en amont est le travail d'une couche que ce moteur n'a pas.

In [11]:
// La seconde insertion remplace les métadonnées à contenu égal.
var ec = new KnowledgeBase();
ec.AddProposition(new("La réunion est utile", confidence: .9, source: "expert"));
ec.AddProposition(new("La réunion est utile", confidence: .1, source: "sondage"));
foreach (var x in ec.Propositions.Values)
    Console.WriteLine($"{x.Content} confidence={x.Confidence} source={x.Source}");
var pa = new Proposition("P", confidence: .9);
var pb = new Proposition("P", confidence: .2);
Console.WriteLine($"Égalité={pa.Equals(pb)}, hashes égaux={pa.GetHashCode() == pb.GetHashCode()}");

La réunion est utile confidence=0.1 source=sondage


Égalité=True, hashes égaux=True


**Lecture du résultat.** Le dictionnaire est indexé par contenu : la seconde écriture écrase la première, et c'est le `sondage` (confidence 0,1) qui survit au `expert` (0,9) — l'ordre d'arrivée décide, pas la qualité. Dans un débat multi-agents où chacun dépose sa version de la même thèse, cela signifie concrètement : **la dernière source qui parle gagne les métadonnées**, sans trace de la précédente. La cause structurelle est en bas de cellule : l'égalité de `Proposition` ignore `confidence`, `truth_value` et `source` ; seul compte le contenu (et le hachage suit, ce qui autorise le dictionnaire). Ce choix a une vertu — une proposition reste unique en base, quel que soit le nombre d'arguments qui la portent, et le décompte du §1 reste stable — et un coût : les métadonnées de la dernière écriture écrasent silencieusement les précédentes.

### Exercice 3 — Prédire avant de tester

Une base contient `« Le débat est vif »` et `« ¬¬Le débat est vif »`. **Avant** d'exécuter la cellule suivante, noter votre prédiction : `IsConsistent` répond-il `True` ou `False`, et `entails(« ¬Le débat est vif »)` ? Puis remplir la cellule pour vérifier.

In [12]:
// Exercice 3 : prédire la cohérence et l'appartenance de ¬P avant de vérifier.
var predit = new KnowledgeBase();
predit.AddProposition(new("Le débat est vif"));
predit.AddProposition(new("¬¬Le débat est vif"));
var predictions = new List<bool>(); // TODO étudiant
Console.WriteLine($"Prédictions : {string.Join(",", predictions)} ; réel : {predit.IsConsistent()}, {predit.Entails(new("¬Le débat est vif"))}");

Prédictions :  ; réel : True, False


## Vérifiez votre compréhension

Que signifie `Entails` dans cette base ? Pourquoi deux propositions de même contenu mais de sources différentes ne produisent-elles qu'une entrée ?

<details><summary>Éléments de réponse</summary>

`Entails` vérifie uniquement l'appartenance lexicale. La base indexe par contenu et la seconde insertion remplace la proposition précédente, y compris ses métadonnées.
</details>

## À retenir

- **`AddArgument` est transitif** : prémisses et conclusion entrent dans la base avec l'argument — la mémoire du débat se peuple par les actes eux-mêmes ;
- **support** = conclusion identique ; **attaque** = conclusion préfixée `¬` — une convention lexicale, simple et vérifiable, qui impose une discipline de forme exacte ;
- **`IsConsistent` détecte le conflit ouvert** (P et `¬P` cohabitants) — c'est lui qui signale qu'un débat a lieu, et lui seul ;
- **`Entails` est un test d'appartenance** : nommage honnête dans la docstring du tronc, attente déductive à ne pas y brancher — pas d'explosion depuis une contradiction, pas de chaînage — mesuré : `entails(premisse) = True` pour une prémisse jamais ajoutée explicitement, `entails(intrus) = False`, et depuis la base incohérente `entails(arbitraire) = False` ;
- **les limites mesurées, portées au grand jour** : `rules`/`preferences` morts non portés, `¬¬P` distinct de `P`, écrasement des propositions homonymes par ordre d'arrivée. Aucune n'est un défaut du port : chacune est une mesure du moteur d'origine.

**Position dans la série** : ce carnet est la *mémoire* du débat — la structure où `Toulmin_Model` dépose ses arguments structurés, où `Schemes_Walton` étiquette leurs schémas, et que les cadres de `Dung_AF_Semantics` lisent pour calculer ce qui survit aux attaques. Les trois niveaux de l'argumentation computationnelle : dire (les actes de langage formalisés), se souvenir (cette base), arbitrer (les sémantiques d'acceptabilité). Un moteur de débat complet boucle les trois — la distillation isole le second pour qu'il soit étudiable seul, déterministe, et sans dépendance.

## Pour aller plus loin

- [Index de la sous-série ArgumentAnalysis](README.md) — la place de la mémoire du débat dans le parcours.
- [Index de la série SymbolicAI](../README.md) — le fil complet.

Le carnet [01 (Toulmin)](Argumentation-01-Toulmin-Model-Python.ipynb) produit les arguments structurés que cette base enregistre ; le carnet [04 (dialogues protocolisés)](Argumentation-04-Dialogues-Protocolises-Python.ipynb) montre les actes de langage qui la peuplent au fil d'un échange ; le carnet [03 (Dung)](Argumentation-03-Dung-AF-Semantics-Python.ipynb) lit ce genre de base pour calculer ce qui survit aux attaques ; le carnet [05c (matrice de richesse)](Argumentation-05c-Formal-Richness-Matrix-Python.ipynb) mesure honnêtement ce qu'un moteur décide réellement.

**Référence** — P. M. Dung (1995). « On the Acceptability of Arguments and its Fundamental Role in Nonmonotonic Reasoning, Logic Programming, and n-Person Games », Artificial Intelligence 77 — le modèle soutien/attaque entre arguments que cette base enregistre et que le carnet 03 arbitre.
